In [1]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [2]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [3]:
import pandas as pd
import numpy as np

In [4]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA

In [5]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [6]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


## EXAMPLES OF BYTE COMPARISONS

In [7]:
w1 = 'abhor'
w2 = 'cleft'
w3 = 'frown'
w1b = byte_encode_words(w1)
w2b = byte_encode_words(w2)
w3b = byte_encode_words(w3)

In [8]:
# no letters in common
w1b & w2b

0

In [9]:
# letters in common
w1b & w3b

147456

In [10]:
# bitwise or
w1b | w2b

673975

In [11]:
# this is the same as directly above
byte_encode_words('abhorcleft')

673975

# BUILD LEVEL 2 THROUGH LEVEL 5

In [12]:
def extend_level(level_masks, level_word_idx, word_byte_array):
    """
    Extend a set of letter-disjoint word groups by one more word.

    level_masks:     1D int array, the union-of-letters bitmask for each
                      group in the current level
    level_word_idx:  2D int array, shape (n_groups, level) - the word
                      indices (into word_byte_array) making up each group
    word_byte_array: 1D int array, bitmask for every word in the vocabulary

    Only extends a group with a word whose index is greater than the last
    word already in that group. This is the key change from the original
    approach: it guarantees every combination of words is built exactly
    once, in ascending index order, instead of once per permutation of the
    words that make it up (which is where most of the original cascade's
    time and memory went - a 5-word group was being rebuilt up to 5! = 120
    times by the time it reached level 5).

    Returns (next_masks, next_word_idx) for the extended level.
    """
    n_words = word_byte_array.shape[0]
    last_idx = level_word_idx[:, -1]

    # True where a group and a candidate word share no letters in common
    no_overlap = (level_masks[:, None] & word_byte_array[None, :]) == 0
    # True where the candidate word's index comes after the last word
    # already used in that group (avoids re-deriving the same combination
    # in a different order)
    after_last = np.arange(n_words)[None, :] > last_idx[:, None]

    valid = no_overlap & after_last
    i, j = np.where(valid)

    next_masks = level_masks[i] | word_byte_array[j]
    next_word_idx = np.hstack([level_word_idx[i], j[:, None]]).astype(np.int32)

    return next_masks, next_word_idx

In [13]:
def extend_level_chunked(level_masks, level_word_idx, word_byte_array, chunk_size=10_000):
    """
    Same as extend_level, but processes level_masks/level_word_idx in
    chunks of `chunk_size` rows at a time, so the (chunk_size x n_words)
    comparison matrix built inside extend_level stays a bounded size
    regardless of how large the current level has grown. Yields
    (masks, word_idx) chunks for non-empty results.
    """
    n_rows = level_masks.shape[0]
    for start in range(0, n_rows, chunk_size):
        stop = start + chunk_size
        chunk_masks, chunk_word_idx = extend_level(
            level_masks[start:stop],
            level_word_idx[start:stop],
            word_byte_array,
        )
        if chunk_masks.size:
            yield chunk_masks, chunk_word_idx


def build_next_level(level_masks, level_word_idx, word_byte_array, chunk_size=10_000):
    """Runs extend_level_chunked to completion and concatenates the result."""
    mask_chunks, idx_chunks = [], []
    for chunk_masks, chunk_word_idx in extend_level_chunked(
        level_masks, level_word_idx, word_byte_array, chunk_size
    ):
        mask_chunks.append(chunk_masks)
        idx_chunks.append(chunk_word_idx)

    if not mask_chunks:
        empty_idx = np.empty((0, level_word_idx.shape[1] + 1), dtype=np.int32)
        return np.empty(0, dtype=np.int32), empty_idx

    return np.concatenate(mask_chunks), np.vstack(idx_chunks)

In [14]:
# tune this down if you hit memory pressure, up if you have headroom and
# want fewer python-level loop iterations
CHUNK_SIZE = 10_000

n_words = word_byte_array.shape[0]
word_byte_array = word_byte_array.astype(np.int32)

# level 1 seed: every single word is its own one-word "group"
l1_masks = word_byte_array.copy()
l1_word_idx = np.arange(n_words, dtype=np.int32).reshape(-1, 1)

l2_masks, l2_word_idx = build_next_level(l1_masks, l1_word_idx, word_byte_array, CHUNK_SIZE)
print('l2:', l2_masks.shape)

l3_masks, l3_word_idx = build_next_level(l2_masks, l2_word_idx, word_byte_array, CHUNK_SIZE)
print('l3:', l3_masks.shape)

l4_masks, l4_word_idx = build_next_level(l3_masks, l3_word_idx, word_byte_array, CHUNK_SIZE)
print('l4:', l4_masks.shape)

l5_masks, l5_word_idx = build_next_level(l4_masks, l4_word_idx, word_byte_array, CHUNK_SIZE)
print('l5:', l5_masks.shape)

l2: (3213696,)
l3: (95866204,)
l4: (26133319,)
l5: (538,)


In [15]:
# each row of l5_word_idx is 5 word indices whose letters are pairwise
# disjoint - i.e. a candidate five-groups-of-five solution.
# NOTE: this assumes word_id_list[i] is the word for word_byte_array[i]
# (same positional indexing) - adjust the lookup if yours differs.
l5_words = np.array(word_id_list)[l5_word_idx]
print('candidate groups found:', l5_words.shape[0])

# sanity check against the known solution from earlier in the notebook
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
outcome_set = set(outcome_words)
found = any(set(row) == outcome_set for row in l5_words.tolist())
print('known solution present:', found)

candidate groups found: 538
known solution present: False
